# Tasks and Limits

The Getting Started drove a robot arm to a target pose under ideal conditions: the target was reachable, joint limits were never approached, and the arm had enough freedom to satisfy the task. In practice, these assumptions rarely hold.

This page introduces tasks and limits, mink’s mechanism for handling the trade-offs that arise when they do not. All examples use the same Panda arm as the quickstart.

**Note on Tasks (soft objectives)**: A task expresses something the robot should try to do. Tasks can be violated when necessary; when multiple tasks conflicts mink resolves them by minimizing a weighted sum of their errors.
- Move an end-effector to a pose (FrameTask)
- Keep joints near a nominal configuration (PostureTask)
- Regulate the center of mass (ComTask)

**Notes on Limits (hard constraints)**: A limit describes a boundary the robot must not cross. Limits are strictly enforced; the solver will never return a solution that violates them.
- Joints position bounds (ConfigurationLimit)
- Velocity bounds (VelocityLimit)
- Collision avoidance (CollisionAvoidanceLimit)

Internally, tasks become quadratic objectives in the QP and limits become inequality constraints.

**Note:** ConfigurationLimit is applied automatically when no custom limits list is provided. All other limits must be added explicitly.

# Common Libraries

In [1]:
import os, mujoco
import numpy as np
from mink import Configuration, SE3, SO3, FrameTask, DampingTask, solve_ik
import matplotlib.pyplot as plt
from tool import do_simulation

# Model specification

In [2]:
dir_path = "/Users/seojin/mink/examples"
model_path = os.path.join(dir_path, "franka_emika_panda/mjx_scene.xml")
model = mujoco.MjModel.from_xml_path(model_path)

# What Goes Wrong Without regularization

A single FrameTask is sufficient when the target is well within the workspace. At the workspace boundary, however, two failure modes emerge.

## Singularities

Consider tracking a target that moves outward, requiring the arm to extend toward its reach limit:

In [42]:
duration = 2.0  # seconds
fps = 60
n_frames = int(duration * fps)
gain = 1.0 - 0.01 ** (1.0 / n_frames)  # ≈ 0.038
dt = 1.0 / fps

configuration = Configuration(model)
configuration.update_from_keyframe("home")
origin_target = configuration.get_transform_frame_to_world("attachment_site", "site")

task = FrameTask(
    frame_name = "attachment_site",
    frame_type = "site",
    position_cost = 1.0,
    orientation_cost = 1.0,
    gain = gain,
)
task.set_target(origin_target)

configuration.data.mocap_pos[0] = origin_target.wxyz_xyz[4:]
configuration.data.mocap_quat[0] = origin_target.wxyz_xyz[:4]

renderer = mujoco.Renderer(configuration.model, height=400, width=600)

# Set target
start_pos = np.array([0.5, 0.0, 0.4])
end_pos = np.array([1, 0.0, 0.4])
orientations = np.array([1,0,0,0])

target_move_outwards = np.linspace(start_pos, end_pos, int(n_frames / 2)) # Move outward
target_move_inwards = np.linspace(end_pos, start_pos, int(n_frames / 2)) # Move inward
targets = np.concatenate((target_move_outwards, target_move_inwards), axis = 0) # Move outward and inward
orientations = np.tile(orientations, (n_frames,1))
targets = np.concatenate((orientations, targets), axis = 1)

target_info = {}
target_info["target"] = targets

do_simulation(configuration, task, renderer, n_frames, dt, target_info, is_render = True)

This browser does not support the video tag.


As the target moves away from the base, the arm must straighten. Near full extension, the elbow approaches a singularity: the Jacobian becomes ill-conditioned, and small task-space errors demand unbounded joint velocities.

Without regularization: the arm becomes unstable near full extension. Note the self-collision between the hand and upper arm.

## Nullspace Drift

The opposite problem occurs when the robot has more degrees of freedom than the task requires. With position-only tracking (orientation_cost=0.0), a 7-DOF arm has four extra degrees of freedom. The solver can move the elbow, rotate the wrist, or reconfigure the arm arbitrarily without affecting the end-effector position.

This is **nullspace drift**: uncotrolled motion in degrees of freedom that do not affect the task. Common symptoms include elbow drift, unnecessary inter-frame joint motion, and non-reproducible poses.

## Adding Regularization

Both failure modes stem from an underconstrained QP. **Regularization** adds secondary objectives that resolve the ambiguity, trading a small amount of tracking accuracy for boundeed, predictable motion.

mink provides two regularization tasks:

- DampingTask
    - behavior: minimizes per-step motion
    - best for: numerical stability, when no preferred pose exists
- PostureTask
    - biases toward a reference configuration
    - best for: manipulation with a preferred pose

Either tasks addresses both singularities and nullspace drift. The following sections demonstrate one natural pairing, but they are interchangeable.

In addition, solve_ik(..., damping=...) provides a lightweight numerical damping term that improves conditioning without adding a task. Per-task lm_damping adds error-dependent Levenberg-Marquardt damping for more conservative behavior near infeasible targets.

# Fixing Singularities

DampingTasks penalizes large values of $ \Delta q $:

In [10]:
task = FrameTask(
    frame_name="attachment_site",
    frame_type="site",
    position_cost=1.0,
    orientation_cost=1.0,
)
task.set_target(target)

# Regularization: prefer smaller joint velocities.
damping_task = DampingTask(model, cost=0.5)

duration = 2.0  # seconds
fps = 60
n_frames = int(duration * fps)
dt = 1.0 / fps

for _ in range(n_frames):
    vel = solve_ik(configuration, [task, damping_task], dt, "daqp")
    configuration.integrate_inplace(vel, dt)